<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/04_Blocking_rebuild.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Rebuild Blocking Indexes (condensed)

This replaces the long exploratory version. It rebuilds exactly the 5 production
index variables (`name_index`, `token_index`, `address_token_index`, `char3_index`,
`rare_two_token_index_10`) plus the `missed_s2_ids` set they depend on, with no
experiments/threshold sweeps/transliteration analysis in between. Results are the
same as before — the ≤10 rare two-token-pair blocker gives ~91.4666% blocking recall.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

import os
import gc
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

CLEANED_DATA_ROOT = os.path.join(
    DRIVE_ROOT,
    "03_Experiments",
    "Cleaned_Data"
)

TRAIN_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "train"
)

TEST_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset",
    "test"
)

BLOCKING_OUTPUT_ROOT = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(BLOCKING_OUTPUT_ROOT, exist_ok=True)

print("Cleaned data:")
print(CLEANED_DATA_ROOT)

print("\nBlocking output:")
print(BLOCKING_OUTPUT_ROOT)


Mounted at /content/drive
Cleaned data:
/content/drive/MyDrive/Amazon ML Challenge 2026/03_Experiments/Cleaned_Data

Blocking output:
/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs


In [27]:
RAW_S1_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source1.tsv"
)

RAW_S2_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source2.tsv"
)
RAW_S3_PATH = os.path.join(
    TRAIN_ROOT,
    "train_source3.tsv"
)



In [26]:
GT_PATH = os.path.join(
    TRAIN_ROOT,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s1_cleaned.tsv"
)

S2_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s2_cleaned.tsv"
)

S3_PATH = os.path.join(
    CLEANED_DATA_ROOT,
    "train_s3_cleaned.tsv")

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype={
        "source1_entity_id": "string",
        "matched_entity_ids": "string"
    }
)

print("Ground truth rows:", len(gt))


Ground truth rows: 2206821


In [4]:
gt_lookup = {}

for row in gt.itertuples(index=False):
    s1_id = row.source1_entity_id
    matched = row.matched_entity_ids

    if pd.isna(matched) or matched == "":
        gt_lookup[s1_id] = set()
    else:
        gt_lookup[s1_id] = {
            x.strip()
            for x in str(matched).split(",")
            if x.strip()
        }

print("Ground truth lookup entries:", len(gt_lookup))


Ground truth lookup entries: 2206821


In [5]:
s1_blocking = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=[
        "entity_id",
        "country_norm",
        "name_core",
        "address_norm"
    ],
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string",
        "address_norm": "string"
    }
)

print(s1_blocking.shape)


(2206821, 4)


In [6]:
NAME_STOPWORDS = {
    "inc", "incorporated", "llc", "ltd", "limited",
    "pvt", "private", "company", "co", "corp",
    "corporation", "group", "services", "partners",
    "holdings", "associates", "center", "india",
    "and", "of"
}

def meaningful_tokens(name):
    if pd.isna(name) or not str(name).strip():
        return []

    tokens = str(name).split()

    return [
        token
        for token in tokens
        if token not in NAME_STOPWORDS and len(token) >= 2
    ]


### 1. Exact country + name index (`name_index`)

In [7]:
from collections import defaultdict

def build_name_index(path):
    index = defaultdict(list)

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country_norm", "name_core"],
        chunksize=200_000,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ):
        chunk = chunk[
            chunk["name_core"].notna() &
            (chunk["name_core"] != "")
        ]

        for country, name, entity_id in zip(
            chunk["country_norm"],
            chunk["name_core"],
            chunk["entity_id"]
        ):
            index[(country, name)].append(entity_id)

        del chunk
        gc.collect()

    return index

print("Building S2 exact-name index...")
name_index = build_name_index(S2_PATH)
print("name_index:", len(name_index))


Building S2 exact-name index...
name_index: 3597772


### 2. Name-token index (`token_index`)

In [8]:
from collections import Counter

name_token_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        name_token_counter.update(set(meaningful_tokens(name)))

    del chunk
    gc.collect()

allowed_name_tokens = {
    token
    for token, freq in name_token_counter.items()
    if 2 <= freq <= 500
}

print("Unique name tokens:", len(name_token_counter))
print("Allowed name tokens:", len(allowed_name_tokens))

del name_token_counter
gc.collect()


Unique name tokens: 810209
Allowed name tokens: 167295


0

In [9]:
token_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    for entity_id, name in zip(chunk["entity_id"], chunk["name_core"]):
        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("token_index:", len(token_index))


token_index: 167295


### 3. Address-token index (`address_token_index`)

In [10]:
address_token_counts = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["address_norm"],
    chunksize=200_000,
    dtype={"address_norm": "string"}
):
    for address in chunk["address_norm"].dropna():
        for token in set(str(address).split()):
            if token:
                address_token_counts[token] += 1
    del chunk
    gc.collect()

MIN_ADDRESS_TOKEN_FREQ = 2
MAX_ADDRESS_TOKEN_FREQ = 500

allowed_address_tokens = {
    token
    for token, count in address_token_counts.items()
    if MIN_ADDRESS_TOKEN_FREQ <= count <= MAX_ADDRESS_TOKEN_FREQ
}

print("Unique address tokens:", len(address_token_counts))
print("Allowed address tokens:", len(allowed_address_tokens))

del address_token_counts
gc.collect()


Unique address tokens: 607900
Allowed address tokens: 404250


0

In [11]:
address_token_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "address_norm"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "address_norm": "string"
    }
):
    for entity_id, address in zip(chunk["entity_id"], chunk["address_norm"]):
        if pd.isna(address):
            continue

        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                address_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("address_token_index:", len(address_token_index))


address_token_index: 404250


### 4. `missed_s2_ids` — true S2 matches not caught by the 3 blockers above

In [12]:
missed_s2_ids = set()

total_true = 0
captured_existing = 0

for row in s1_blocking.itertuples(index=False):
    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    true_ids = {
        x
        for x in gt_lookup.get(s1_id, set())
        if x.startswith("S2-")
    }

    if not true_ids:
        continue

    total_true += len(true_ids)

    candidate_set = set()

    if pd.notna(name) and name != "":
        candidate_set.update(name_index.get((country, name), []))

        for token in set(meaningful_tokens(name)):
            if token in allowed_name_tokens:
                candidate_set.update(token_index.get(token, []))

    if pd.notna(address) and address != "":
        for token in set(str(address).split()):
            if token in allowed_address_tokens:
                candidate_set.update(address_token_index.get(token, []))

    captured = true_ids.intersection(candidate_set)

    captured_existing += len(captured)

    missed_s2_ids.update(true_ids - captured)

print("Total true S2 matches:", total_true)
print("Captured by existing blocker:", captured_existing)
print("Missed by existing blocker:", len(missed_s2_ids))


Total true S2 matches: 3693619
Captured by existing blocker: 3222405
Missed by existing blocker: 471214


### 5. Character 3-gram index (`char3_index`) — conservative version, freq 2–500

In [13]:
def char_ngrams(text, n=3):
    if pd.isna(text):
        return set()

    text = str(text).replace(" ", "")

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }

char3_counter = Counter()

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):
    for name in chunk["name_core"].dropna():
        char3_counter.update(char_ngrams(name, n=3))
    del chunk
    gc.collect()

allowed_char3 = {
    gram
    for gram, freq in char3_counter.items()
    if 2 <= freq <= 500
}

print("Unique 3-grams:", len(char3_counter))
print("Allowed 3-grams:", len(allowed_char3))

del char3_counter
gc.collect()


Unique 3-grams: 90023
Allowed 3-grams: 65473


0

In [14]:
char3_index = defaultdict(list)

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    for entity_id, name in zip(chunk["entity_id"], chunk["name_core"]):
        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):
            if gram in allowed_char3:
                char3_index[gram].append(entity_id)

    del chunk
    gc.collect()

print("char3_index:", len(char3_index))


char3_index: 65473


### 6. Rare two-token-pair index (`rare_two_token_index_10`), freq ≤ 10

In [15]:
missed_s2_names = {}

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):
    mask = chunk["entity_id"].isin(missed_s2_ids)

    for entity_id, name in zip(
        chunk.loc[mask, "entity_id"],
        chunk.loc[mask, "name_core"]
    ):
        missed_s2_names[entity_id] = name

    del chunk
    gc.collect()

print("Missed S2 names:", len(missed_s2_names))


Missed S2 names: 471214


In [16]:
from itertools import combinations

def name_token_pairs(name):
    tokens = sorted(set(meaningful_tokens(name)))

    if len(tokens) < 2:
        return set()

    return set(combinations(tokens, 2))

two_token_frequency = defaultdict(int)

for name in missed_s2_names.values():
    if pd.isna(name):
        continue
    for pair in name_token_pairs(name):
        two_token_frequency[pair] += 1

print("Unique two-token pairs:", len(two_token_frequency))


Unique two-token pairs: 362476


In [17]:
rare_two_token_index_10 = defaultdict(list)

for s2_id, name in missed_s2_names.items():
    if pd.isna(name):
        continue

    for pair in name_token_pairs(name):
        if two_token_frequency[pair] <= 10:
            rare_two_token_index_10[pair].append(s2_id)

print("rare_two_token_index_10:", len(rare_two_token_index_10))
print(
    "Indexed references:",
    sum(len(v) for v in rare_two_token_index_10.values())
)


rare_two_token_index_10: 353966
Indexed references: 522162


### Verify everything

In [18]:
print("name_index:", len(name_index))
print("token_index:", len(token_index))
print("address_token_index:", len(address_token_index))
print("char3_index:", len(char3_index))
print("rare_two_token_index_10:", len(rare_two_token_index_10))


name_index: 3597772
token_index: 167295
address_token_index: 404250
char3_index: 65473
rare_two_token_index_10: 353966


In [19]:
import os
import gc
import pandas as pd

CANDIDATE_OUTPUT_DIR = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs"
)

os.makedirs(CANDIDATE_OUTPUT_DIR, exist_ok=True)

CANDIDATE_PATH = os.path.join(
    CANDIDATE_OUTPUT_DIR,
    "train_candidate_pairs.tsv"
)

# Remove an old partial output if it exists
if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print("Output:", CANDIDATE_PATH)

Output: /content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/train_candidate_pairs.tsv


In [22]:
# =========================================================
# BUILD SELECTIVE NAME-TOKEN INDEX
# =========================================================

NAME_TOKEN_MAX_CANDIDATES = 100

selective_token_index = {}

for token, ids in token_index.items():

    if len(ids) <= NAME_TOKEN_MAX_CANDIDATES:
        selective_token_index[token] = ids

print("Original name-token keys:", len(token_index))
print("Selective name-token keys:", len(selective_token_index))

print(
    "Original indexed references:",
    sum(len(v) for v in token_index.values())
)

print(
    "Selective indexed references:",
    sum(len(v) for v in selective_token_index.values())
)

Original name-token keys: 167295
Selective name-token keys: 161071
Original indexed references: 3230167
Selective indexed references: 1999717


In [23]:
# =========================================================
# BUILD SELECTIVE ADDRESS-TOKEN INDEX
# =========================================================

ADDRESS_TOKEN_MAX_CANDIDATES = 100

selective_address_token_index = {}

for token, ids in address_token_index.items():

    if len(ids) <= ADDRESS_TOKEN_MAX_CANDIDATES:
        selective_address_token_index[token] = ids

print("Original address-token keys:", len(address_token_index))
print(
    "Selective address-token keys:",
    len(selective_address_token_index)
)

print(
    "Original indexed references:",
    sum(len(v) for v in address_token_index.values())
)

print(
    "Selective indexed references:",
    sum(
        len(v)
        for v in selective_address_token_index.values()
    )
)

Original address-token keys: 404250
Selective address-token keys: 389029
Original indexed references: 6812664
Selective indexed references: 3568962


In [24]:
# =========================================================
# TEST CANDIDATE VOLUME ON 50K S1
# =========================================================

TEST_ROWS = 50_000

test_s1 = s1_blocking.iloc[:TEST_ROWS]

candidate_counts = []

for row in test_s1.itertuples(index=False):

    s1_id = row.entity_id
    country = row.country_norm
    name = row.name_core
    address = row.address_norm

    candidates = set()

    # 1. Exact country + name
    if pd.notna(country) and pd.notna(name):
        candidates.update(
            name_index.get((country, name), [])
        )

    # 2. Selective name token
    if pd.notna(name):

        for token in meaningful_tokens(name):

            candidates.update(
                selective_token_index.get(token, [])
            )

    # 3. Selective address token
    if pd.notna(address):

        for token in set(str(address).split()):

            candidates.update(
                selective_address_token_index.get(token, [])
            )

    # 4. Conservative 3-gram
    if pd.notna(name):

        for gram in char_ngrams(name, n=3):

            candidates.update(
                char3_index.get(gram, [])
            )

    # 5. Rare two-token pair <=10
    if pd.notna(name):

        for pair in name_token_pairs(name):

            candidates.update(
                rare_two_token_index_10.get(pair, [])
            )

    candidate_counts.append(len(candidates))

candidate_counts = pd.Series(candidate_counts)

print("====================================")
print("50K CANDIDATE VOLUME TEST")
print("====================================")
print("Mean:", candidate_counts.mean())
print("Median:", candidate_counts.median())
print("P75:", candidate_counts.quantile(0.75))
print("P90:", candidate_counts.quantile(0.90))
print("P95:", candidate_counts.quantile(0.95))
print("P99:", candidate_counts.quantile(0.99))
print("Max:", candidate_counts.max())
print("Total:", candidate_counts.sum())
print(
    "Estimated 2.2M candidates:",
    int(
        candidate_counts.mean()
        * len(s1_blocking)
    )
)

50K CANDIDATE VOLUME TEST
Mean: 98.1712
Median: 48.0
P75: 104.0
P90: 279.0
P95: 428.0
P99: 714.0
Max: 1944
Total: 4908560
Estimated 2.2M candidates: 216646265


In [28]:
# =========================================================
# S3 BASELINE: COUNTRY + EXACT NAME
# =========================================================

s3_name_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "country_norm", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
):

    for entity_id, country, name in zip(
        chunk["entity_id"],
        chunk["country_norm"],
        chunk["name_core"]
    ):

        if pd.isna(country) or pd.isna(name):
            continue

        s3_name_index[(country, name)].append(entity_id)

    del chunk
    gc.collect()

print("S3 name index:", len(s3_name_index))

S3 name index: 3850257


In [30]:
# =========================================================
# FINAL SELECTIVE S2 NAME-TOKEN INDEX
# =========================================================

S2_NAME_MAX_FREQ = 10

s2_selective_name_index = {
    token: ids
    for token, ids in token_index.items()
    if len(ids) <= S2_NAME_MAX_FREQ
}

print("S2 selective name keys:", len(s2_selective_name_index))
print(
    "S2 selective references:",
    sum(len(v) for v in s2_selective_name_index.values())
)

S2 selective name keys: 112966
S2 selective references: 432065


In [31]:
# =========================================================
# BUILD FINAL S3 BLOCKING INDEXES
# =========================================================

from collections import defaultdict
import pandas as pd
import gc

S3_NAME_MAX_FREQ = 10

# ---------------------------------------------------------
# A. S3 exact country + name
# ---------------------------------------------------------

s3_name_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "country_norm", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "country_norm": "string",
        "name_core": "string"
    }
):

    for entity_id, country, name in zip(
        chunk["entity_id"],
        chunk["country_norm"],
        chunk["name_core"]
    ):
        if pd.isna(country) or pd.isna(name):
            continue

        s3_name_index[(country, name)].append(entity_id)

    del chunk
    gc.collect()

print("S3 exact-name keys:", len(s3_name_index))


# ---------------------------------------------------------
# B. S3 name-token frequency
# ---------------------------------------------------------

s3_token_freq = defaultdict(int)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):

    for name in chunk["name_core"]:

        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):
            s3_token_freq[token] += 1

    del chunk
    gc.collect()

s3_allowed_tokens = {
    token
    for token, freq in s3_token_freq.items()
    if 2 <= freq <= S3_NAME_MAX_FREQ
}

print("S3 total tokens:", len(s3_token_freq))
print("S3 selective tokens:", len(s3_allowed_tokens))

del s3_token_freq
gc.collect()


# ---------------------------------------------------------
# C. S3 selective name-token index
# ---------------------------------------------------------

s3_token_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for token in set(meaningful_tokens(name)):

            if token in s3_allowed_tokens:
                s3_token_index[token].append(entity_id)

    del chunk
    gc.collect()

print("S3 selective token keys:", len(s3_token_index))


# ---------------------------------------------------------
# D. S3 3-gram frequencies
# ---------------------------------------------------------

s3_char3_freq = defaultdict(int)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["name_core"],
    chunksize=200_000,
    dtype={"name_core": "string"}
):

    for name in chunk["name_core"]:

        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):
            s3_char3_freq[gram] += 1

    del chunk
    gc.collect()

s3_allowed_char3 = {
    gram
    for gram, freq in s3_char3_freq.items()
    if 2 <= freq <= 500
}

print("S3 allowed 3-grams:", len(s3_allowed_char3))

del s3_char3_freq
gc.collect()


# ---------------------------------------------------------
# E. S3 3-gram index
# ---------------------------------------------------------

s3_char3_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for gram in char_ngrams(name, n=3):

            if gram in s3_allowed_char3:
                s3_char3_index[gram].append(entity_id)

    del chunk
    gc.collect()

print("S3 3-gram keys:", len(s3_char3_index))

print("\nS3 indexes ready.")

S3 exact-name keys: 3850257
S3 total tokens: 862709
S3 selective tokens: 130540
S3 selective token keys: 130540
S3 allowed 3-grams: 66673
S3 3-gram keys: 66673

S3 indexes ready.


In [ ]:
# =========================================================
# BUILD S3 RARE TWO-TOKEN-PAIR INDEX
# =========================================================

s3_pair_freq = defaultdict(int)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for name in chunk["name_core"]:

        if pd.isna(name):
            continue

        for pair in name_token_pairs(name):
            s3_pair_freq[pair] += 1

    del chunk
    gc.collect()

print("S3 unique token pairs:", len(s3_pair_freq))


s3_rare_pair_index = defaultdict(list)

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=["entity_id", "name_core"],
    chunksize=200_000,
    dtype={
        "entity_id": "string",
        "name_core": "string"
    }
):

    for entity_id, name in zip(
        chunk["entity_id"],
        chunk["name_core"]
    ):

        if pd.isna(name):
            continue

        for pair in name_token_pairs(name):

            if s3_pair_freq[pair] <= 10:
                s3_rare_pair_index[pair].append(entity_id)

    del chunk
    gc.collect()

print("S3 rare pair keys:", len(s3_rare_pair_index))
print(
    "S3 rare pair references:",
    sum(len(v) for v in s3_rare_pair_index.values())
)

del s3_pair_freq
gc.collect()

S3 unique token pairs: 4336863


In [33]:
import os

CANDIDATE_PATH = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs",
    "candidate_pairs.tsv"
)

if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print(CANDIDATE_PATH)

/content/drive/MyDrive/Amazon ML Challenge 2026/05_Outputs/Candidate_Pairs/candidate_pairs.tsv


In [34]:
# =========================================================
# FINAL S2 + S3 CANDIDATE GENERATOR
# =========================================================

import os
import gc
import pandas as pd

CANDIDATE_PATH = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs",
    "candidate_pairs.tsv"
)

if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

CHUNK_SIZE = 25_000

# Safety cap per S1/source.
# We collect candidates in blocker priority order.
MAX_CANDIDATES = 200

first_write = True
total_pairs = 0
total_s1 = 0

# ---------------------------------------------------------
# Helper: generate candidates for one source
# ---------------------------------------------------------

def generate_source_candidates(
    s1_row,
    exact_index,
    selective_token_index,
    gram_index,
    source_prefix,
    max_candidates=200
):

    s1_id = s1_row.entity_id
    country = s1_row.country_norm
    name = s1_row.name_core

    candidates = set()

    # -----------------------------------------------------
    # 1. Exact country + name
    # -----------------------------------------------------
    if pd.notna(country) and pd.notna(name):

        candidates.update(
            exact_index.get(
                (country, name),
                []
            )
        )

    # -----------------------------------------------------
    # 2. Rare/selective name tokens
    # -----------------------------------------------------
    if len(candidates) < max_candidates and pd.notna(name):

        for token in meaningful_tokens(name):

            ids = selective_token_index.get(token)

            if ids:
                candidates.update(ids)

            if len(candidates) >= max_candidates:
                break

    # -----------------------------------------------------
    # 3. Character 3-grams
    # -----------------------------------------------------
    if len(candidates) < max_candidates and pd.notna(name):

        for gram in char_ngrams(name, n=3):

            ids = gram_index.get(gram)

            if ids:
                candidates.update(ids)

            if len(candidates) >= max_candidates:
                break

    # -----------------------------------------------------
    # Restrict to cap
    # -----------------------------------------------------
    if len(candidates) > max_candidates:
        candidates = set(
            list(candidates)[:max_candidates]
        )

    return [
        (s1_id, candidate_id, source_prefix)
        for candidate_id in candidates
    ]


# =========================================================
# PROCESS S1
# =========================================================

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S1_PATH,
        sep="\t",
        usecols=[
            "entity_id",
            "country_norm",
            "name_core",
            "address_norm"
        ],
        chunksize=CHUNK_SIZE,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string",
            "address_norm": "string"
        }
    ),
    start=1
):

    rows = []

    for row in chunk.itertuples(index=False):

        # -------------------------------------------------
        # S2
        # -------------------------------------------------

        rows.extend(
            generate_source_candidates(
                row,
                name_index,
                s2_selective_name_index,
                char3_index,
                "S2",
                MAX_CANDIDATES
            )
        )

        # -------------------------------------------------
        # S3
        # -------------------------------------------------

        rows.extend(
            generate_source_candidates(
                row,
                s3_name_index,
                s3_token_index,
                s3_char3_index,
                "S3",
                MAX_CANDIDATES
            )
        )

    if rows:

        out = pd.DataFrame(
            rows,
            columns=[
                "source1_entity_id",
                "candidate_entity_id",
                "source"
            ]
        )

        out.drop_duplicates(
            inplace=True
        )

        out.to_csv(
            CANDIDATE_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        total_pairs += len(out)

        del out
        del rows

    total_s1 += len(chunk)

    if chunk_no % 4 == 0:
        print(
            f"S1 processed: {total_s1:,} | "
            f"Candidate pairs: {total_pairs:,}"
        )

    del chunk
    gc.collect()

    first_write = False


print("\n========================================")
print("CANDIDATE GENERATION COMPLETE")
print("========================================")
print("S1 processed:", f"{total_s1:,}")
print("Candidate pairs:", f"{total_pairs:,}")
print("Output:", CANDIDATE_PATH)

S1 processed: 100,000 | Candidate pairs: 7,505,278
S1 processed: 200,000 | Candidate pairs: 14,964,363
S1 processed: 300,000 | Candidate pairs: 22,465,329
S1 processed: 400,000 | Candidate pairs: 29,864,162
S1 processed: 500,000 | Candidate pairs: 37,375,314
S1 processed: 600,000 | Candidate pairs: 44,891,833
S1 processed: 700,000 | Candidate pairs: 52,409,632
S1 processed: 800,000 | Candidate pairs: 59,878,387
S1 processed: 900,000 | Candidate pairs: 67,375,376
S1 processed: 1,000,000 | Candidate pairs: 74,901,244
S1 processed: 1,100,000 | Candidate pairs: 82,463,258
S1 processed: 1,200,000 | Candidate pairs: 89,989,197


KeyboardInterrupt: 

In [35]:
import os

if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

print("Partial candidate file removed.")

Partial candidate file removed.


In [36]:
# =========================================================
# 10K CANDIDATE VOLUME TEST — SIMPLIFIED
# =========================================================

TEST_ROWS = 10_000

test_s1 = s1_blocking.iloc[:TEST_ROWS]

s2_counts = []
s3_counts = []
total_counts = []

for row in test_s1.itertuples(index=False):

    name = row.name_core
    country = row.country_norm

    # =====================================================
    # S2
    # =====================================================

    s2_candidates = set()

    # Exact country + name
    if pd.notna(country) and pd.notna(name):
        s2_candidates.update(
            name_index.get(
                (country, name),
                []
            )
        )

    # Rare 2-token pair <=10
    if pd.notna(name):

        for pair in name_token_pairs(name):
            s2_candidates.update(
                rare_two_token_index_10.get(pair, [])
            )

    # =====================================================
    # S3
    # =====================================================

    s3_candidates = set()

    # Exact country + name
    if pd.notna(country) and pd.notna(name):
        s3_candidates.update(
            s3_name_index.get(
                (country, name),
                []
            )
        )

    # Selective name tokens <=10
    if pd.notna(name):

        for token in meaningful_tokens(name):

            ids = s3_token_index.get(token)

            if ids:
                s3_candidates.update(ids)

    # =====================================================
    # Record statistics
    # =====================================================

    s2_counts.append(len(s2_candidates))
    s3_counts.append(len(s3_candidates))
    total_counts.append(
        len(s2_candidates) +
        len(s3_candidates)
    )


s2_counts = pd.Series(s2_counts)
s3_counts = pd.Series(s3_counts)
total_counts = pd.Series(total_counts)

print("========================================")
print("SIMPLIFIED 10K CANDIDATE TEST")
print("========================================")

print("\nS2")
print("Mean:", s2_counts.mean())
print("Median:", s2_counts.median())
print("P90:", s2_counts.quantile(.90))
print("P95:", s2_counts.quantile(.95))
print("P99:", s2_counts.quantile(.99))
print("Max:", s2_counts.max())
print("Total:", s2_counts.sum())

print("\nS3")
print("Mean:", s3_counts.mean())
print("Median:", s3_counts.median())
print("P90:", s3_counts.quantile(.90))
print("P95:", s3_counts.quantile(.95))
print("P99:", s3_counts.quantile(.99))
print("Max:", s3_counts.max())
print("Total:", s3_counts.sum())

print("\nCOMBINED")
print("Mean:", total_counts.mean())
print("Median:", total_counts.median())
print("P90:", total_counts.quantile(.90))
print("P95:", total_counts.quantile(.95))
print("P99:", total_counts.quantile(.99))
print("Max:", total_counts.max())
print("Total:", total_counts.sum())

estimated_total = (
    total_counts.mean()
    * len(s1_blocking)
)

print(
    "\nEstimated full candidate pairs:",
    f"{estimated_total:,.0f}"
)

SIMPLIFIED 10K CANDIDATE TEST

S2
Mean: 16.8712
Median: 3.0
P90: 30.0
P95: 75.0
P99: 262.0100000000002
Max: 647
Total: 168712

S3
Mean: 16.6818
Median: 2.0
P90: 38.0
P95: 71.0
P99: 259.0100000000002
Max: 584
Total: 166818

COMBINED
Mean: 33.553
Median: 5.0
P90: 65.0
P95: 145.04999999999927
P99: 513.0
Max: 1231
Total: 335530

Estimated full candidate pairs: 74,045,465


In [37]:
MAX_CANDIDATES_PER_SOURCE = 50

s2_counts = []
s3_counts = []

for row in test_s1.itertuples(index=False):

    name = row.name_core
    country = row.country_norm

    # -----------------------------
    # S2
    # -----------------------------
    s2_candidates = set()

    if pd.notna(country) and pd.notna(name):
        s2_candidates.update(
            name_index.get((country, name), [])
        )

    if pd.notna(name):
        for pair in name_token_pairs(name):
            s2_candidates.update(
                rare_two_token_index_10.get(pair, [])
            )

    if len(s2_candidates) > MAX_CANDIDATES_PER_SOURCE:
        s2_candidates = set(
            list(s2_candidates)[:MAX_CANDIDATES_PER_SOURCE]
        )

    # -----------------------------
    # S3
    # -----------------------------
    s3_candidates = set()

    if pd.notna(country) and pd.notna(name):
        s3_candidates.update(
            s3_name_index.get((country, name), [])
        )

    if pd.notna(name):
        for token in meaningful_tokens(name):
            ids = s3_token_index.get(token)
            if ids:
                s3_candidates.update(ids)

    if len(s3_candidates) > MAX_CANDIDATES_PER_SOURCE:
        s3_candidates = set(
            list(s3_candidates)[:MAX_CANDIDATES_PER_SOURCE]
        )

    s2_counts.append(len(s2_candidates))
    s3_counts.append(len(s3_candidates))


s2_counts = pd.Series(s2_counts)
s3_counts = pd.Series(s3_counts)

print("S2 mean:", s2_counts.mean())
print("S2 P95:", s2_counts.quantile(.95))
print("S2 max:", s2_counts.max())

print("S3 mean:", s3_counts.mean())
print("S3 P95:", s3_counts.quantile(.95))
print("S3 max:", s3_counts.max())

print(
    "Estimated combined pairs:",
    int(
        (s2_counts.mean() + s3_counts.mean())
        * len(s1_blocking)
    )
)

S2 mean: 9.755
S2 P95: 50.0
S2 max: 50
S3 mean: 9.7391
S3 P95: 50.0
S3 max: 50
Estimated combined pairs: 43019989


In [ ]:
# =========================================================
# FINAL CANDIDATE GENERATION — S2 + S3
# =========================================================

import os
import gc
import pandas as pd

CANDIDATE_PATH = os.path.join(
    DRIVE_ROOT,
    "05_Outputs",
    "Candidate_Pairs",
    "candidate_pairs.tsv"
)

if os.path.exists(CANDIDATE_PATH):
    os.remove(CANDIDATE_PATH)

CHUNK_SIZE = 25_000
MAX_CANDIDATES_PER_SOURCE = 50

first_write = True
total_pairs = 0
total_s1 = 0


def get_candidates(
    country,
    name,
    exact_index,
    token_index,
    fuzzy_index,
    rare_pair_index=None,
    max_candidates=50
):
    """
    Candidate priority:
      1. Exact country + name
      2. Rare two-token pair, if supplied
      3. Selective name tokens
      4. 3-grams

    Exact candidates are NEVER displaced by fuzzy candidates.
    """

    candidates = set()

    # -----------------------------------------------------
    # 1. EXACT COUNTRY + NAME
    # -----------------------------------------------------

    if pd.notna(country) and pd.notna(name):

        candidates.update(
            exact_index.get(
                (country, name),
                []
            )
        )

    # -----------------------------------------------------
    # 2. RARE TWO-TOKEN PAIR
    # -----------------------------------------------------

    if (
        rare_pair_index is not None
        and pd.notna(name)
        and len(candidates) < max_candidates
    ):

        for pair in name_token_pairs(name):

            ids = rare_pair_index.get(pair)

            if ids:
                candidates.update(ids)

            if len(candidates) >= max_candidates:
                break

    # -----------------------------------------------------
    # 3. SELECTIVE NAME TOKEN
    # -----------------------------------------------------

    if (
        pd.notna(name)
        and len(candidates) < max_candidates
    ):

        for token in meaningful_tokens(name):

            ids = token_index.get(token)

            if ids:
                candidates.update(ids)

            if len(candidates) >= max_candidates:
                break

    # -----------------------------------------------------
    # 4. CHARACTER 3-GRAM
    # -----------------------------------------------------

    if (
        pd.notna(name)
        and len(candidates) < max_candidates
    ):

        for gram in char_ngrams(name, n=3):

            ids = fuzzy_index.get(gram)

            if ids:
                candidates.update(ids)

            if len(candidates) >= max_candidates:
                break

    # -----------------------------------------------------
    # FINAL CAP
    # -----------------------------------------------------

    if len(candidates) > max_candidates:

        candidates = set(
            list(candidates)[:max_candidates]
        )

    return candidates


# =========================================================
# PROCESS S1
# =========================================================

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S1_PATH,
        sep="\t",
        usecols=[
            "entity_id",
            "country_norm",
            "name_core"
        ],
        chunksize=CHUNK_SIZE,
        dtype={
            "entity_id": "string",
            "country_norm": "string",
            "name_core": "string"
        }
    ),
    start=1
):

    rows = []

    for row in chunk.itertuples(index=False):

        s1_id = row.entity_id
        country = row.country_norm
        name = row.name_core

        # =================================================
        # S2
        # =================================================

        s2_candidates = get_candidates(
            country=country,
            name=name,
            exact_index=name_index,
            token_index=s2_selective_name_index,
            fuzzy_index=char3_index,
            rare_pair_index=rare_two_token_index_10,
            max_candidates=MAX_CANDIDATES_PER_SOURCE
        )

        rows.extend(
            (
                s1_id,
                s2_id,
                "S2"
            )
            for s2_id in s2_candidates
        )

        # =================================================
        # S3
        # =================================================

        s3_candidates = get_candidates(
            country=country,
            name=name,
            exact_index=s3_name_index,
            token_index=s3_token_index,
            fuzzy_index=s3_char3_index,
            rare_pair_index=None,
            max_candidates=MAX_CANDIDATES_PER_SOURCE
        )

        rows.extend(
            (
                s1_id,
                s3_id,
                "S3"
            )
            for s3_id in s3_candidates
        )

    # -----------------------------------------------------
    # WRITE CHUNK
    # -----------------------------------------------------

    if rows:

        out = pd.DataFrame(
            rows,
            columns=[
                "source1_entity_id",
                "candidate_entity_id",
                "source"
            ]
        )

        out.drop_duplicates(
            inplace=True
        )

        out.to_csv(
            CANDIDATE_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        total_pairs += len(out)

        del out
        del rows

    total_s1 += len(chunk)

    if chunk_no % 4 == 0:

        print(
            f"S1 processed: {total_s1:,} | "
            f"Candidate pairs: {total_pairs:,}"
        )

    del chunk
    gc.collect()

    first_write = False


print("\n========================================")
print("CANDIDATE GENERATION COMPLETE")
print("========================================")
print(
    "S1 processed:",
    f"{total_s1:,}"
)
print(
    "Candidate pairs:",
    f"{total_pairs:,}"
)
print(
    "Output:",
    CANDIDATE_PATH
)